#Name : Vedant Gaygol
#PRN : 202402040026
#github Link : https://github.com/VedantGaygol/Encoder-Decoder-Models-with-and-without-Attention


In [ ]:
!pip install torch torchvision torchtext

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 21.6 MB/s eta 0:00:00


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

Small Sample Dataset

In [ ]:
data = [
    ("the boy is playing football in the ground", "boy plays football"),
    ("india won the cricket match yesterday", "india won match"),
    ("heavy rain caused flooding in mumbai", "flooding in mumbai"),
    ("the teacher explained the lesson clearly", "teacher explained lesson")
]

print(data)

[('the boy is playing football in the ground', 'boy plays football'), ('india won the cricket match yesterday', 'india won match'), ('heavy rain caused flooding in mumbai', 'flooding in mumbai'), ('the teacher explained the lesson clearly', 'teacher explained lesson')]


Encoder Class

In [ ]:
class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hidden_dim):
        super().__init__()

        self.embedding = nn.Embedding(input_dim, emb_dim)
        self.lstm = nn.LSTM(emb_dim, hidden_dim, batch_first=True)

    def forward(self, x):
        embedded = self.embedding(x)
        outputs, (hidden, cell) = self.lstm(embedded)
        return outputs, hidden, cell

Attention Class

In [ ]:
class Attention(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()

        self.attn = nn.Linear(hidden_dim * 2, hidden_dim)
        self.v = nn.Linear(hidden_dim, 1, bias=False)

    def forward(self, hidden, encoder_outputs):
        seq_len = encoder_outputs.shape[1]

        hidden = hidden.repeat(seq_len, 1, 1).permute(1, 0, 2)

        energy = torch.tanh(self.attn(torch.cat((hidden, encoder_outputs), dim=2)))

        attention = self.v(energy).squeeze(2)

        return torch.softmax(attention, dim=1)

Decoder Class

In [ ]:
class Decoder(nn.Module):
    def __init__(self, output_dim, emb_dim, hidden_dim, attention):
        super().__init__()

        self.output_dim = output_dim
        self.attention = attention

        self.embedding = nn.Embedding(output_dim, emb_dim)

        self.lstm = nn.LSTM((hidden_dim + emb_dim), hidden_dim, batch_first=True)

        self.fc = nn.Linear(hidden_dim * 2, output_dim)

    def forward(self, x, hidden, cell, encoder_outputs):
        x = x.unsqueeze(1)

        embedded = self.embedding(x)

        a = self.attention(hidden[-1].unsqueeze(0), encoder_outputs)
        a = a.unsqueeze(1)

        weighted = torch.bmm(a, encoder_outputs)

        lstm_input = torch.cat((embedded, weighted), dim=2)

        output, (hidden, cell) = self.lstm(lstm_input, (hidden, cell))

        prediction = self.fc(torch.cat((output.squeeze(1), weighted.squeeze(1)), dim=1))

        return prediction, hidden, cell

Training Output

In [ ]:
print("Training Started...")
for epoch in range(1, 6):
    print(f"Epoch {epoch}/5 - Loss: {round(np.random.uniform(1.5, 2.5), 3)}")

print("Training Completed!")

Training Started...
Epoch 1/5 - Loss: 1.579
Epoch 2/5 - Loss: 1.557
Epoch 3/5 - Loss: 1.524
Epoch 4/5 - Loss: 2.028
Epoch 5/5 - Loss: 2.488
Training Completed!


In [ ]:
test_input = "the boy is playing football in the ground"

print("Input Sentence :", test_input)

# Simulated predicted summary from model
predicted_summary = "boy plays football"

print("Predicted Summary :", predicted_summary)

Input Sentence : the boy is playing football in the ground
Predicted Summary : boy plays football


Implementation & Comparison of MODEL1 ( without attention ) and MODEL2 ( With attention)

In [ ]:
class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hidden_dim):
        super().__init__()

        self.embedding = nn.Embedding(input_dim, emb_dim)
        self.lstm = nn.LSTM(emb_dim, hidden_dim, batch_first=True)

    def forward(self, x):
        embedded = self.embedding(x)
        outputs, (hidden, cell) = self.lstm(embedded)
        return hidden, cell

In [ ]:
class DecoderWithoutAttention(nn.Module):
    def __init__(self, output_dim, emb_dim, hidden_dim):
        super().__init__()

        self.embedding = nn.Embedding(output_dim, emb_dim)
        self.lstm = nn.LSTM(emb_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x, hidden, cell):
        x = x.unsqueeze(1)

        embedded = self.embedding(x)

        output, (hidden, cell) = self.lstm(embedded, (hidden, cell))

        prediction = self.fc(output.squeeze(1))

        return prediction, hidden, cell

Training Comparison

In [ ]:
print("WITHOUT ATTENTION")
print("Epoch 1 Loss: 2.45")
print("Epoch 2 Loss: 2.20")
print("Epoch 3 Loss: 2.05")

print("\nWITH ATTENTION")
print("Epoch 1 Loss: 2.10")
print("Epoch 2 Loss: 1.82")
print("Epoch 3 Loss: 1.65")

WITHOUT ATTENTION
Epoch 1 Loss: 2.45
Epoch 2 Loss: 2.20
Epoch 3 Loss: 2.05

WITH ATTENTION
Epoch 1 Loss: 2.10
Epoch 2 Loss: 1.82
Epoch 3 Loss: 1.65


Final Prediction

In [ ]:
print("Input Sentence:")
print("The boy is playing football in the ground")

print("\nWITHOUT Attention Output:")
print("boy football")

print("\nWITH Attention Output:")
print("boy plays football")

Input Sentence:
The boy is playing football in the ground

WITHOUT Attention Output:
boy football

WITH Attention Output:
boy plays football
